# 🛡️ Steganalysis — Full Training Notebook (Colab Ready)

This notebook trains the complete Multi-Branch CNN Steganalysis model.

**Run this on Google Colab (free T4 GPU) for Month 2-3 training.**

---
**Step 1:** Upload your project folder to Google Drive

**Step 2:** Run all cells in order

**Step 3:** Download the `weights/` folder to your laptop when done

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# IMPORTANT: Change this path to where you uploaded the project on Drive
PROJECT_PATH = '/content/drive/MyDrive/steganalysis_project'
import os
os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 2: Install Dependencies ───────────────────────────────────────────────
!pip install -q torch torchvision pillow opencv-python scipy scikit-learn \
               matplotlib seaborn tqdm pandas streamlit

In [ ]:
# ── Cell 3: Verify GPU ─────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
else:
    print('⚠️  No GPU detected. Training will be SLOW on CPU.')
    print('   Tip: Runtime → Change runtime type → GPU → T4')

In [ ]:
# ── Cell 4: Setup Dataset ──────────────────────────────────────────────────────
# If BOSS dataset is already in data/raw/, run the setup script:
!bash scripts/setup_boss_dataset.sh

# Check dataset
import os
clean_count = len(os.listdir('data/clean'))
stego_count = len(os.listdir('data/stego'))
print(f'Clean: {clean_count} images')
print(f'Stego: {stego_count} images')
assert clean_count == stego_count, 'Dataset imbalanced! Check setup.'
print('✅ Dataset balanced (50/50 clean/stego)')

In [ ]:
# ── Cell 5: Train Branch A (Pixel CNN) ────────────────────────────────────────
# Student A: Run this in Week 5-6
# Expected time: ~1-2 hours on Colab T4 GPU with 10,000 images

!python src/training/train.py \
    --mode branch_a \
    --epochs 40 \
    --batch_size 64 \
    --lr 1e-4

print('\n✅ Branch A training complete!')
print('Checkpoint saved to: weights/branch_a_best.pt')

In [ ]:
# ── Cell 6: Train Branch B (DCT Frequency CNN) ────────────────────────────────
# Student B: Run this in Week 5-6 (parallel to Student A on their machine)

!python src/training/train.py \
    --mode branch_b \
    --epochs 40 \
    --batch_size 64 \
    --lr 1e-4

In [ ]:
# ── Cell 7: Train Branch C (Statistics MLP) ───────────────────────────────────
# Both students, Week 7
# NOTE: Branch C uses CPU-side feature extraction — slightly slower per batch

!python src/training/train.py \
    --mode branch_c \
    --epochs 30 \
    --batch_size 32

In [ ]:
# ── Cell 8: Train Full Fusion Model ───────────────────────────────────────────
# Both students pair-programming, Weeks 9-10
# This loads Branch A+B+C weights and fine-tunes everything together

!python src/training/train.py \
    --mode fusion \
    --epochs 50 \
    --batch_size 32 \
    --lr 5e-5

In [ ]:
# ── Cell 9: Evaluate + Ablation Study ─────────────────────────────────────────
# Week 11 — generates comparison table and plots

!python src/training/evaluate.py --ablation

# Plot training history
!python src/training/evaluate.py \
    --history results/history_fusion.json

print('\nResults saved to: results/')
import os
print('Files:', os.listdir('results/'))

In [ ]:
# ── Cell 10: Show Results Plots ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, fname, title in [
    (axes[0], 'results/roc_curves.png', 'ROC Curves'),
    (axes[1], 'results/training_history.png', 'Training History'),
]:
    if os.path.exists(fname):
        img = mpimg.imread(fname)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: INT8 Quantization ────────────────────────────────────────────────
# Week 11 — compress model for CPU deployment

!python src/deployment/quantize.py \
    --checkpoint weights/fusion_best.pt \
    --output weights/fusion_int8.pt \
    --compare

# Benchmark speed
!python src/deployment/inference.py \
    --checkpoint weights/fusion_int8.pt \
    --int8 \
    --benchmark

print('\nCopy the numbers above into your paper Table of Results!')

In [ ]:
# ── Cell 12: Download Results ─────────────────────────────────────────────────
# Download weights and results to your local machine

import shutil
from google.colab import files

# Zip up weights and results
shutil.make_archive('steganalysis_results', 'zip', '.', 'weights')
shutil.make_archive('steganalysis_plots',   'zip', '.', 'results')

files.download('steganalysis_results.zip')
files.download('steganalysis_plots.zip')

print('✅ Files downloaded!')
print('Unzip and place weights/ in your project folder.')